# Comparação com modelo baseado em árvores

Este notebook compara o baseline interpretável de regressão logística com um modelo baseado em árvores.

A intenção não é substituir automaticamente o baseline. A comparação serve para avaliar se um modelo mais flexível captura padrões adicionais no dataset sintético e se esse ganho compensa a perda de simplicidade.

Os dados são sintéticos. Os resultados não representam desempenho real em produção.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH_CANDIDATES = [
    Path("data/customers_churn_synthetic.csv"),
    Path("../data/customers_churn_synthetic.csv"),
]

DATA_PATH = next(path for path in DATA_PATH_CANDIDATES if path.exists())

df = pd.read_csv(DATA_PATH)

df.head()


,customer_id,tenure_months,access_technology,download_speed_mbps,monthly_fee,has_contract_loyalty,overdue_invoice_count,oldest_overdue_days,active_agreement_installment_amount,had_price_adjustment_90d,support_tickets_90d,repeat_issue_90d,avg_resolution_hours_90d,outage_count_30d,network_outage_hours_30d,churn_90d
0,CUST-000001,40,fiber,500,160.93,0,0,0,0.00,1,0,0,0.00,0,0.00,0
1,CUST-000002,53,fiber,200,109.99,1,0,0,0.00,0,0,0,0.00,0,0.00,0
2,CUST-000003,35,fiber,100,100.73,0,1,46,0.00,0,0,0,0.00,0,0.00,0
3,CUST-000004,31,fiber,200,116.91,1,0,0,61.27,0,0,0,0.00,1,1.13,1
4,CUST-000005,58,fiber,100,105.92,1,2,32,0.00,0,1,0,47.23,0,0.00,0


## Preparação da base

Vamos usar as mesmas variáveis e a mesma lógica de separação do baseline.

Isso deixa a comparação mais justa: se os modelos recebem os mesmos dados de treino e teste, a diferença observada vem principalmente da estratégia de modelagem.


In [2]:
from sklearn.model_selection import train_test_split

TARGET = "churn_90d"
FEATURES = [column for column in df.columns if column not in ["customer_id", TARGET]]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

summary = pd.DataFrame(
    {
        "base": ["treino", "teste"],
        "linhas": [len(X_train), len(X_test)],
        "taxa_churn": [y_train.mean(), y_test.mean()],
    }
)
summary["taxa_churn"] = summary["taxa_churn"].mul(100).round(2)

summary


,base,linhas,taxa_churn
0,treino,3750,10.35
1,teste,1250,10.32


## Pré-processamento

A regressão logística usa padronização das variáveis numéricas porque seus coeficientes são sensíveis à escala.

A Random Forest não precisa de padronização numérica, porque divide os dados por cortes nas variáveis. Para ela, vamos apenas transformar a variável categórica `access_technology` em indicadores.


In [3]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = ["access_technology"]
numeric_features = [column for column in FEATURES if column not in categorical_features]

logistic_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), numeric_features),
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ]
)

tree_preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numeric_features),
        ("categorical", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features),
    ]
)

print(f"Variáveis numéricas: {len(numeric_features)}")
print(f"Variáveis categóricas: {len(categorical_features)}")


Variáveis numéricas: 13
Variáveis categóricas: 1


## Modelos comparados

Vamos comparar três modelos:

- `DummyClassifier`: referência ingênua que prevê sempre a classe majoritária.
- Regressão logística balanceada: baseline interpretável já construído.
- Random Forest balanceada: modelo baseado em árvores, capaz de capturar interações e relações não lineares.

A Random Forest foi configurada com profundidade limitada e número mínimo de amostras por folha para reduzir overfitting e manter a comparação mais estável.


In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

models = {
    "dummy_classe_majoritaria": DummyClassifier(strategy="most_frequent"),
    "logistica_balanced": Pipeline(
        steps=[
            ("preprocessor", logistic_preprocessor),
            ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced")),
        ]
    ),
    "random_forest_balanced": Pipeline(
        steps=[
            ("preprocessor", tree_preprocessor),
            (
                "classifier",
                RandomForestClassifier(
                    n_estimators=300,
                    max_depth=6,
                    min_samples_leaf=20,
                    class_weight="balanced",
                    random_state=42,
                    n_jobs=-1,
                ),
            ),
        ]
    ),
}

for model in models.values():
    model.fit(X_train, y_train)

print("Modelos treinados com sucesso.")


Modelos treinados com sucesso.


## Métricas de comparação

A comparação mantém as mesmas métricas usadas no baseline.

Acurácia continua sendo uma métrica secundária, porque churn é minoritário. Para o negócio, recall, precision, F1, ROC-AUC, Average Precision e quantidade de clientes sinalizados são mais úteis.


In [5]:
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)


def evaluate_model(name, fitted_model):
    y_pred = fitted_model.predict(X_test)
    y_score = fitted_model.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    return {
        "modelo": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
        "Average Precision": average_precision_score(y_test, y_score),
        "clientes_sinalizados": int(y_pred.sum()),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

comparison = pd.DataFrame(
    [evaluate_model(name, model) for name, model in models.items()]
)

comparison.round(4)


,modelo,accuracy,precision,recall,f1,roc_auc,Average Precision,clientes_sinalizados,tn,fp,fn,tp
0,dummy_classe_majoritaria,0.8968,0.0000,0.0000,0.0000,0.5000,0.1032,0,1121,0,129,0
1,logistica_balanced,0.6184,0.1345,0.4961,0.2116,0.6069,0.1808,476,709,412,65,64
2,random_forest_balanced,0.7080,0.1509,0.3953,0.2184,0.6006,0.1377,338,834,287,78,51


## Cenários operacionais por ranking

Além do `predict` com limiar padrão, vamos avaliar a utilidade do score como ranking de priorização.

Os cenários abaixo simulam capacidade de abordagem da equipe de retenção: top 10%, 20% e 30% dos clientes com maior score. Esses percentuais não são limiares otimizados no conjunto de teste.


In [6]:
def evaluate_capacity_scenario(model_name, fitted_model, share):
    scores = fitted_model.predict_proba(X_test)[:, 1]
    ranking = pd.DataFrame(
        {
            "churn_90d": y_test.to_numpy(),
            "score": scores,
        }
    ).sort_values("score", ascending=False)

    prioritized_count = int(np.ceil(len(ranking) * share))
    prioritized = ranking.head(prioritized_count)
    churns_found = int(prioritized["churn_90d"].sum())
    total_churns = int(ranking["churn_90d"].sum())
    precision = churns_found / prioritized_count
    recall = churns_found / total_churns
    lift = precision / ranking["churn_90d"].mean()

    return {
        "modelo": model_name,
        "cenário": f"top {int(share * 100)}%",
        "clientes_priorizados": prioritized_count,
        "churns_encontrados": churns_found,
        "precision": precision,
        "recall": recall,
        "lift_vs_taxa_geral": lift,
    }

capacity_comparison = pd.DataFrame(
    [
        evaluate_capacity_scenario(model_name, model, share)
        for model_name, model in models.items()
        if model_name != "dummy_classe_majoritaria"
        for share in [0.10, 0.20, 0.30]
    ]
)

capacity_comparison.round(4)


,modelo,cenário,clientes_priorizados,churns_encontrados,precision,recall,lift_vs_taxa_geral
0,logistica_balanced,top 10%,125,24,0.1920,0.1860,1.8605
1,logistica_balanced,top 20%,250,38,0.1520,0.2946,1.4729
2,logistica_balanced,top 30%,375,52,0.1387,0.4031,1.3437
3,random_forest_balanced,top 10%,125,18,0.1440,0.1395,1.3953
4,random_forest_balanced,top 20%,250,40,0.1600,0.3101,1.5504
5,random_forest_balanced,top 30%,375,59,0.1573,0.4574,1.5245


## Importância das variáveis na Random Forest

A Random Forest oferece uma medida de importância das variáveis baseada no uso das variáveis nas árvores.

Essa leitura indica quais variáveis ajudaram mais o modelo a separar os grupos neste dataset sintético. Ela não demonstra causalidade e também pode ser influenciada por correlações entre variáveis.


In [7]:
random_forest_model = models["random_forest_balanced"]
feature_names = random_forest_model.named_steps["preprocessor"].get_feature_names_out()
feature_names = [
    name.replace("numeric__", "").replace("categorical__", "")
    for name in feature_names
]

feature_importance = pd.DataFrame(
    {
        "variável": feature_names,
        "importância": random_forest_model.named_steps["classifier"].feature_importances_,
    }
).sort_values("importância", ascending=False)

feature_importance.head(10).round(4)


,variável,importância
10,avg_resolution_hours_90d,0.2066
2,monthly_fee,0.1226
5,oldest_overdue_days,0.1184
0,tenure_months,0.1015
12,network_outage_hours_30d,0.0950
8,support_tickets_90d,0.0786
3,has_contract_loyalty,0.0667
1,download_speed_mbps,0.0571
4,overdue_invoice_count,0.0361
7,had_price_adjustment_90d,0.0322


## Conclusões da comparação

A comparação deve ser lida em duas dimensões:

- desempenho técnico: se o modelo baseado em árvores melhora métricas como ROC-AUC, Average Precision, recall e lift nos cenários de capacidade;
- utilidade prática: se o ganho compensa a perda de simplicidade em relação à regressão logística.

Mesmo que a Random Forest apresente melhora, ela ainda não torna o projeto pronto para produção. Antes de uso real, seria necessário validar com dados reais, usar separação temporal, calibrar probabilidades, avaliar custos de abordagem e definir governança operacional.

O próximo passo natural é transformar a comparação em uma leitura executiva: quais métricas importam para retenção, qual cenário de capacidade parece mais realista e como traduzir o ranking em uma fila de ação.
